In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

df = pd.read_csv('../../data/raw/customer_churn_dataset-testing-master.csv')

In [3]:
print("RAW DATA LOADED")
print(f"Total Records: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
print(f"\nColumn Names:\n{df.columns.tolist()}")

RAW DATA LOADED
Total Records: 64,374
Total Columns: 12

Column Names:
['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction', 'Churn']


#### INITIAL DATA EXPLORATION

In [4]:
print("DATA TYPES")
print("-"*30)
print(df.dtypes)

DATA TYPES
------------------------------
CustomerID            int64
Age                   int64
Gender               object
Tenure                int64
Usage Frequency       int64
Support Calls         int64
Payment Delay         int64
Subscription Type    object
Contract Length      object
Total Spend           int64
Last Interaction      int64
Churn                 int64
dtype: object


In [5]:
print("SAMPLE ROWS")
print("-"*70)
print(df.head(10))

SAMPLE ROWS
----------------------------------------------------------------------
   CustomerID  Age  Gender  Tenure  Usage Frequency  Support Calls  \
0           1   22  Female      25               14              4   
1           2   41  Female      28               28              7   
2           3   47    Male      27               10              2   
3           4   35    Male       9               12              5   
4           5   53  Female      58               24              9   
5           6   30    Male      41               14             10   
6           7   47  Female      37               15              9   
7           8   54  Female      36               11              0   
8           9   36    Male      20                5             10   
9          10   65    Male       8                4              2   

   Payment Delay Subscription Type Contract Length  Total Spend  \
0             27             Basic         Monthly          598   
1           

In [6]:
print("STATISTICAL SUMMARY")
print("-"*70)
print(df.describe())

STATISTICAL SUMMARY
----------------------------------------------------------------------
         CustomerID           Age        Tenure  Usage Frequency  \
count  64374.000000  64374.000000  64374.000000     64374.000000   
mean   32187.500000     41.970982     31.994827        15.080234   
std    18583.317451     13.924911     17.098234         8.816470   
min        1.000000     18.000000      1.000000         1.000000   
25%    16094.250000     30.000000     18.000000         7.000000   
50%    32187.500000     42.000000     33.000000        15.000000   
75%    48280.750000     54.000000     47.000000        23.000000   
max    64374.000000     65.000000     60.000000        30.000000   

       Support Calls  Payment Delay   Total Spend  Last Interaction  \
count   64374.000000   64374.000000  64374.000000      64374.000000   
mean        5.400690      17.133952    541.023379         15.498850   
std         3.114005       8.852211    260.874809          8.638436   
min         

In [7]:
print("DATA QUALITY CHECK")
print("-"*25)
print(f"Duplicate Customer IDs: {df['CustomerID'].duplicated().sum()}")
print(f"Missing Values: {df.isnull().sum().sum()}")

DATA QUALITY CHECK
-------------------------
Duplicate Customer IDs: 0
Missing Values: 0


#### Feature Engineering - MRR (Monthly Recurring Revenue)

In [8]:
print("\n" + "="*70)
print("ENGINEERING FEATURE 3: CLV (Customer Lifetime Value)")
print("="*70)

# Clean column names (important safety step)
df.columns = df.columns.str.strip()

# ----------------------------
# STEP 1: Create MRR proxy
# ----------------------------
# Since dataset has NO MRR, we derive it from Total Spend

df['MRR'] = df['Total Spend'] / df['Tenure'].replace(0, 1)

# ----------------------------
# STEP 2: Actual CLV (historical revenue)
# ----------------------------
df['CLV_Actual'] = df['Total Spend']

# ----------------------------
# STEP 3: Churn rate & expected lifetime
# ----------------------------
overall_churn_rate = df['Churn'].mean()

expected_lifetime_months = (
    1 / overall_churn_rate if overall_churn_rate > 0 else 0
)

# ----------------------------
# STEP 4: Predicted CLV
# ----------------------------
df['CLV_Predicted'] = df['MRR'] * expected_lifetime_months

# ----------------------------
# STEP 5: Lost CLV (only for churned customers)
# ----------------------------
df['CLV_Lost'] = np.where(
    df['Churn'] == 1,
    df['CLV_Predicted'] - df['CLV_Actual'],
    0
)

# ----------------------------
# OUTPUT
# ----------------------------
print(f"\nOverall Churn Rate: {overall_churn_rate:.2%}")
print(f"Expected Customer Lifetime: {expected_lifetime_months:.1f} months")

print("\n" + "="*70)
print("CLV ANALYSIS")
print("="*70)

print(df[['CustomerID', 'Churn', 'MRR', 'Tenure',
          'CLV_Actual', 'CLV_Predicted', 'CLV_Lost']].head(15))

# ----------------------------
# TOTAL LOSS
# ----------------------------
total_revenue_lost = df['CLV_Lost'].sum()
print(f"\n💰 Total Predicted Revenue Lost to Churn: ${total_revenue_lost:,.2f}")

# ----------------------------
# GROUP ANALYSIS
# ----------------------------
print("\n" + "="*70)
print("CLV BY SUBSCRIPTION TYPE")
print("="*70)

clv_by_type = df.groupby('Subscription Type').agg({
    'CLV_Actual': 'mean',
    'CLV_Predicted': 'mean',
    'CLV_Lost': 'sum'
}).round(2)

print(clv_by_type)


ENGINEERING FEATURE 3: CLV (Customer Lifetime Value)

Overall Churn Rate: 47.37%
Expected Customer Lifetime: 2.1 months

CLV ANALYSIS
    CustomerID  Churn         MRR  Tenure  CLV_Actual  CLV_Predicted  \
0            1      1   23.920000      25         598      50.497691   
1            2      0   20.857143      28         584      44.031670   
2            3      0   28.037037      27         757      59.189198   
3            4      0   25.777778       9         232      54.419659   
4            5      0    9.189655      58         533      19.400350   
5            6      0   12.195122      41         500      25.745213   
6            7      1   15.513514      37         574      32.750694   
7            8      0    8.972222      36         323      18.941325   
8            9      0   34.350000      20         687      72.516542   
9           10      0  124.375000       8         995     262.568991   
10          11      1   12.523810      42         526      26.439108   
1

In [9]:
def assign_cohort(tenure):
    if tenure <= 3:
        return 'New (0-3m)'
    elif tenure <= 12:
        return 'Growing (4-12m)'
    elif tenure <= 24:
        return 'Mature (13-24m)'
    else:
        return 'Veteran (25m+)'

df['Cohort'] = df['Tenure'].apply(assign_cohort)


print("\nCohort Distribution:")
print(df['Cohort'].value_counts().sort_index())


print("\n" + "="*70)
print("CHURN RATE BY COHORT")
print("="*70)
cohort_churn = df.groupby('Cohort').agg({
    'Churn': ['count', 'sum', 'mean']
}).round(4)
cohort_churn.columns = ['Total Customers', 'Churned', 'Churn Rate']
cohort_churn['Churn Rate %'] = (cohort_churn['Churn Rate'] * 100).round(2)
print(cohort_churn)


print("\n BUSINESS INSIGHT:")
highest_churn_cohort = cohort_churn['Churn Rate %'].idxmax()
print(f"Highest churn in: {highest_churn_cohort}")
print("This tells us WHEN to intervene with retention campaigns.")


Cohort Distribution:
Cohort
Growing (4-12m)     8514
Mature (13-24m)    11287
New (0-3m)          2846
Veteran (25m+)     41727
Name: count, dtype: int64

CHURN RATE BY COHORT
                 Total Customers  Churned  Churn Rate  Churn Rate %
Cohort                                                             
Growing (4-12m)             8514     2621      0.3078         30.78
Mature (13-24m)            11287     3546      0.3142         31.42
New (0-3m)                  2846      995      0.3496         34.96
Veteran (25m+)             41727    23331      0.5591         55.91

 BUSINESS INSIGHT:
Highest churn in: Veteran (25m+)
This tells us WHEN to intervene with retention campaigns.


In [12]:
print(" CUSTOMER HEALTH SCORE")
print("-"*70)

# RISK SIGNALS:
# 1. Payment Delay (high = bad)
# 2. Support Calls (high = struggling)
# 3. Usage Frequency (low = disengaged)
# 4. Tenure (low = not established)


# : Normalize each metric to 0-100 scale
# ----------------------------

# Payment Delay Score (0 = many delays, 100 = no delays)
max_payment_delay = df['Payment Delay'].max()
df['Score_Payment'] = 100 - ((df['Payment Delay'] / max_payment_delay) * 100)

# Support Calls Score (0 = many calls, 100 = few calls)
max_support_calls = df['Support Calls'].max()
df['Score_Support'] = 100 - ((df['Support Calls'] / max_support_calls) * 100)

# Usage Frequency Score (0 = low usage, 100 = high usage)
max_usage = df['Usage Frequency'].max()
df['Score_Usage'] = (df['Usage Frequency'] / max_usage) * 100

# Tenure Score (0 = new customer, 100 = veteran)
max_tenure = df['Tenure'].max()
df['Score_Tenure'] = (df['Tenure'] / max_tenure) * 100


# : Weighted Health Score
# ----------------------------

# Different metrics have different importance
# Weights (must sum to 1.0):
# - Payment: 30% (most critical - indicates financial issues)
# - Support: 25% (indicates product struggles)
# - Usage: 30% (indicates engagement)
# - Tenure: 15% (indicates loyalty, but less predictive than behavior)

df['Health_Score'] = (
    (df['Score_Payment'] * 0.30) +
    (df['Score_Support'] * 0.25) +
    (df['Score_Usage'] * 0.30) +
    (df['Score_Tenure'] * 0.15)
)


# : Health Score Categories
# ----------------------------
def categorize_health(score):
    """Categorize customer health into risk segments"""
    if score >= 75:
        return 'Healthy'
    elif score >= 50:
        return 'At Risk'
    elif score >= 25:
        return 'Critical'
    else:
        return 'Churning'

df['Health_Category'] = df['Health_Score'].apply(categorize_health)


# OUTPUT
# ----------------------------
print("\nHealth Score Distribution:")
print(df['Health_Category'].value_counts().sort_index())

print("\n" + "-"*70)
print("HEALTH SCORE BY CHURN STATUS")
print("="*70)
health_churn = df.groupby(['Health_Category', 'Churn']).size().unstack(fill_value=0)
health_churn['Churn_Rate_%'] = (health_churn[1] / (health_churn[0] + health_churn[1]) * 100).round(2)
print(health_churn)

print("\n" + "-"*70)
print("SAMPLE CUSTOMERS WITH HEALTH SCORES")
print("="*70)
print(df[['CustomerID', 'Payment Delay', 'Support Calls', 'Usage Frequency', 
          'Tenure', 'Health_Score', 'Health_Category', 'Churn']].head(20))

# BUSINESS INSIGHT
print("\n BUSINESS INSIGHT:")
critical_customers = len(df[df['Health_Category'].isin(['Critical', 'Churning'])])
print(f" {critical_customers:,} customers ({critical_customers/len(df)*100:.1f}%) are in Critical/Churning status")
print("These should be prioritized for immediate retention outreach.")

 CUSTOMER HEALTH SCORE
----------------------------------------------------------------------

Health Score Distribution:
Health_Category
At Risk     24661
Churning     4289
Critical    32622
Healthy      2802
Name: count, dtype: int64

----------------------------------------------------------------------
HEALTH SCORE BY CHURN STATUS
Churn                0      1  Churn_Rate_%
Health_Category                            
At Risk          18542   6119         24.81
Churning           848   3441         80.23
Critical         11694  20928         64.15
Healthy           2797      5          0.18

----------------------------------------------------------------------
SAMPLE CUSTOMERS WITH HEALTH SCORES
    CustomerID  Payment Delay  Support Calls  Usage Frequency  Tenure  \
0            1             27              4               14      25   
1            2             13              7               28      28   
2            3             29              2               10      27   

In [27]:
print(" USAGE ENGAGEMENT TIERS")
print("-"*70)

# WHY ENGAGEMENT TIERS MATTER:
# "Usage Frequency" is a number (4, 5, 10, etc), but what does it MEAN?
# We need to bucket users into actionable segments


# : Understand the distribution
# ----------------------------
print("Usage Frequency Statistics:")
print(df['Usage Frequency'].describe())

# Calculate quartiles
q25 = df['Usage Frequency'].quantile(0.25)
q75 = df['Usage Frequency'].quantile(0.75)
median = df['Usage Frequency'].median()

print(f"\n25th Percentile: {q25}")
print(f"Median: {median}")
print(f"75th Percentile: {q75}")


# : Create engagement tiers
# ----------------------------
# We'll use quartile-based segmentation:
# - Low: Bottom 25%
# - Medium: 25-75%
# - High: Top 25%

def assign_engagement_tier(usage):
    if usage <= q25:
        return 'Low'
    elif usage <= q75:
        return 'Medium'
    else:
        return 'High'

df['Engagement_Tier'] = df['Usage Frequency'].apply(assign_engagement_tier)





 USAGE ENGAGEMENT TIERS
----------------------------------------------------------------------
Usage Frequency Statistics:
count    64374.000000
mean        15.080234
std          8.816470
min          1.000000
25%          7.000000
50%         15.000000
75%         23.000000
max         30.000000
Name: Usage Frequency, dtype: float64

25th Percentile: 7.0
Median: 15.0
75th Percentile: 23.0


In [26]:
print("ENGAGEMENT TIER DISTRIBUTION")
print("-"*30)
print(df['Engagement_Tier'].value_counts())


print("-"*30)
print("CHURN RATE BY ENGAGEMENT TIER")
print("-"*30)
engagement_churn = df.groupby('Engagement_Tier')['Churn'].agg(['count', 'sum', 'mean'])
engagement_churn.columns = ['Total', 'Churned', 'Churn_Rate']
engagement_churn['Churn_Rate_%'] = (engagement_churn['Churn_Rate'] * 100).round(2)
print(engagement_churn)

ENGAGEMENT TIER DISTRIBUTION
------------------------------
Engagement_Tier
Medium    33251
Low       16638
High      14485
Name: count, dtype: int64
------------------------------
CHURN RATE BY ENGAGEMENT TIER
------------------------------
                 Total  Churned  Churn_Rate  Churn_Rate_%
Engagement_Tier                                          
High             14485     6256    0.431895         43.19
Low              16638     9900    0.595023         59.50
Medium           33251    14337    0.431175         43.12


In [23]:
# Cross-analysis: Engagement + Health

print("ENGAGEMENT TIER × HEALTH CATEGORY")
print("-"*55)
cross_tab = pd.crosstab(df['Engagement_Tier'], df['Health_Category'])
print(cross_tab)

print("\nBUSINESS INSIGHT:")
print("-"*55)
low_engagement_churn = df[df['Engagement_Tier'] == 'Low']['Churn'].mean() * 100
high_engagement_churn = df[df['Engagement_Tier'] == 'High']['Churn'].mean() * 100
print(f"Low Engagement Churn Rate: {low_engagement_churn:.1f}%")
print(f"High Engagement Churn Rate: {high_engagement_churn:.1f}%")
print(f"Difference: {low_engagement_churn - high_engagement_churn:.1f} percentage points")

ENGAGEMENT TIER × HEALTH CATEGORY
-------------------------------------------------------
Health_Category  At Risk  Churning  Critical  Healthy
Engagement_Tier                                      
High                9044         0      3565     1876
Low                 2521      3138     10978        1
Medium             13096      1151     18079      925

BUSINESS INSIGHT:
-------------------------------------------------------
Low Engagement Churn Rate: 59.5%
High Engagement Churn Rate: 43.2%
Difference: 16.3 percentage points


In [29]:
print("\n" + "="*70)
print("ENGINEERING FEATURE 7: PAYMENT RISK FLAGS")
print("="*70)

# WHY PAYMENT RISK MATTERS:
# Payment issues are the #1 churn predictor
# We need to flag customers with payment problems BEFORE they churn

# ----------------------------
# STEP 1: Define risk thresholds
# ----------------------------
print("Payment Delay Distribution:")
print(df['Payment Delay'].value_counts().sort_index())

# Risk levels:
# - No Risk: 0 delays
# - Low Risk: 1-2 delays
# - High Risk: 3+ delays

def assign_payment_risk(delays):
    """Categorize payment risk based on number of delays"""
    if delays == 0:
        return 'No Risk'
    elif delays <= 2:
        return 'Low Risk'
    else:
        return 'High Risk'

df['Payment_Risk'] = df['Payment Delay'].apply(assign_payment_risk)

# Binary flag for immediate action
df['Payment_Risk_Flag'] = (df['Payment Delay'] >= 3).astype(int)

# ----------------------------
# OUTPUT
# ----------------------------
print("\n" + "="*70)
print("PAYMENT RISK DISTRIBUTION")
print("="*70)
print(df['Payment_Risk'].value_counts())

print("\n" + "="*70)
print("CHURN RATE BY PAYMENT RISK")
print("="*70)
payment_churn = df.groupby('Payment_Risk')['Churn'].agg(['count', 'sum', 'mean'])
payment_churn.columns = ['Total', 'Churned', 'Churn_Rate']
payment_churn['Churn_Rate_%'] = (payment_churn['Churn_Rate'] * 100).round(2)
print(payment_churn)

# High-risk customers with high CLV (save these first!)
print("\n" + "="*70)
print("HIGH-VALUE CUSTOMERS AT PAYMENT RISK")
print("="*70)
high_risk_high_value = df[
    (df['Payment_Risk'] == 'High Risk') & 
    (df['CLV_Actual'] > df['CLV_Actual'].median()) &
    (df['Churn'] == 0)  # Haven't churned yet
].sort_values('CLV_Actual', ascending=False)

print(f"Found {len(high_risk_high_value)} high-value customers at risk")
print("\nTop 10:")
print(high_risk_high_value[['CustomerID', 'Payment Delay', 'CLV_Actual', 
                             'Subscription Type', 'Health_Score']].head(10))

print("\n ACTION ITEM:")
print("These customers should receive immediate outreach (payment plan, discount, etc)")


ENGINEERING FEATURE 7: PAYMENT RISK FLAGS
Payment Delay Distribution:
Payment Delay
0     1594
1     1539
2     1531
3     1543
4     1588
5     1568
6     1541
7     1539
8     1559
9     1534
10    1646
11    1589
12    1587
13    1608
14    1618
15    1586
16    2363
17    2337
18    2284
19    2261
20    2280
21    2766
22    2821
23    2716
24    2748
25    2781
26    2781
27    2818
28    2726
29    2748
30    2774
Name: count, dtype: int64

PAYMENT RISK DISTRIBUTION
Payment_Risk
High Risk    59710
Low Risk      3070
No Risk       1594
Name: count, dtype: int64

CHURN RATE BY PAYMENT RISK
              Total  Churned  Churn_Rate  Churn_Rate_%
Payment_Risk                                          
High Risk     59710    30015    0.502680         50.27
Low Risk       3070      325    0.105863         10.59
No Risk        1594      153    0.095985          9.60

HIGH-VALUE CUSTOMERS AT PAYMENT RISK
Found 15995 high-value customers at risk

Top 10:
       CustomerID  Payment Delay  

In [30]:
print("\n" + "="*70)
print(" INTERACTION RECENCY")
print("="*70)

# WHY RECENCY MATTERS:
# "Last Interaction" tells us how recently the customer engaged
# Recent activity = engaged, Old activity = ghosting us

# ----------------------------
# STEP 1: Understand the data
# ----------------------------
print("Last Interaction Distribution:")
print(df['Last Interaction'].describe())

# Calculate recency quartiles
q33 = df['Last Interaction'].quantile(0.33)
q66 = df['Last Interaction'].quantile(0.66)

print(f"\n33rd Percentile: {q33}")
print(f"66th Percentile: {q66}")

# ----------------------------
# STEP 2: Create recency categories
# ----------------------------
def assign_recency_category(days):
    """Categorize interaction recency (lower days = more recent = better)"""
    if days <= q33:
        return 'Recent'
    elif days <= q66:
        return 'Moderate'
    else:
        return 'Stale'

df['Interaction_Recency'] = df['Last Interaction'].apply(assign_recency_category)

# Reverse score (lower days = higher score)
max_last_interaction = df['Last Interaction'].max()
df['Recency_Score'] = 100 - ((df['Last Interaction'] / max_last_interaction) * 100)

# ----------------------------
# OUTPUT
# ----------------------------
print("\n" + "="*70)
print("INTERACTION RECENCY DISTRIBUTION")
print("="*70)
print(df['Interaction_Recency'].value_counts())

print("\n" + "="*70)
print("CHURN RATE BY INTERACTION RECENCY")
print("="*70)
recency_churn = df.groupby('Interaction_Recency')['Churn'].agg(['count', 'sum', 'mean'])
recency_churn.columns = ['Total', 'Churned', 'Churn_Rate']
recency_churn['Churn_Rate_%'] = (recency_churn['Churn_Rate'] * 100).round(2)
print(recency_churn)

# Dormant customers (stale + haven't churned yet)
print("\n" + "="*70)
print("DORMANT CUSTOMERS (At Risk)")
print("="*70)
dormant = df[
    (df['Interaction_Recency'] == 'Stale') & 
    (df['Churn'] == 0)
]
print(f"Found {len(dormant):,} dormant customers who haven't churned yet")
print(f"Potential revenue at risk: ${dormant['MRR'].sum():,.2f}/month")

print("\n📊 BUSINESS INSIGHT:")
print("These customers should receive re-engagement campaigns:")
print("- Email: 'We miss you! Here's what's new...'")
print("- Offer: Limited-time discount or feature unlock")
print("- Survey: 'Why haven't you been using us?'")


 INTERACTION RECENCY
Last Interaction Distribution:
count    64374.000000
mean        15.498850
std          8.638436
min          1.000000
25%          8.000000
50%         15.000000
75%         23.000000
max         30.000000
Name: Last Interaction, dtype: float64

33rd Percentile: 10.0
66th Percentile: 20.0

INTERACTION RECENCY DISTRIBUTION
Interaction_Recency
Moderate    21591
Recent      21422
Stale       21361
Name: count, dtype: int64

CHURN RATE BY INTERACTION RECENCY
                     Total  Churned  Churn_Rate  Churn_Rate_%
Interaction_Recency                                          
Moderate             21591    10121    0.468760         46.88
Recent               21422    10237    0.477873         47.79
Stale                21361    10135    0.474463         47.45

DORMANT CUSTOMERS (At Risk)
Found 11,226 dormant customers who haven't churned yet
Potential revenue at risk: $512,096.25/month

📊 BUSINESS INSIGHT:
These customers should receive re-engagement campaigns:
- 

In [31]:
print("\n" + "="*70)
print("FINAL DATA VALIDATION & EXPORT")
print("="*70)

# ----------------------------
# STEP 1: Check for any issues
# ----------------------------
print("1. Checking for missing values in engineered features...")
engineered_cols = ['MRR', 'CLV_Actual', 'CLV_Predicted', 'Health_Score', 
                   'Engagement_Tier', 'Payment_Risk', 'Interaction_Recency', 'Cohort']
missing_check = df[engineered_cols].isnull().sum()
print(missing_check[missing_check > 0] if missing_check.sum() > 0 else "✅ No missing values")

print("\n2. Checking for infinite/invalid values...")
inf_check = np.isinf(df[['MRR', 'CLV_Actual', 'CLV_Predicted']]).sum()
print("✅ No infinite values" if inf_check.sum() == 0 else f"⚠️ Found infinite values:\n{inf_check}")

print("\n3. Final dataset shape:")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

# ----------------------------
# STEP 2: Create analysis-ready dataset
# ----------------------------
print("\n" + "="*70)
print("CREATING ANALYSIS-READY DATASET")
print("="*70)

# Select final columns for analysis
final_columns = [
    # Identifiers
    'CustomerID',
    
    # Original features
    'Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls',
    'Payment Delay', 'Subscription Type', 'Contract Length', 
    'Total Spend', 'Last Interaction', 'Churn',
    
    # Engineered features
    'MRR', 'CLV_Actual', 'CLV_Predicted', 'CLV_Lost',
    'Health_Score', 'Health_Category',
    'Engagement_Tier', 'Payment_Risk', 'Payment_Risk_Flag',
    'Interaction_Recency', 'Recency_Score', 'Cohort'
]

df_final = df[final_columns].copy()

# ----------------------------
# STEP 3: Export to CSV
# ----------------------------
output_path = '../../data/processed/churn_data_engineered.csv'
df_final.to_csv(output_path, index=False)
print(f"\n✅ Saved engineered dataset to: {output_path}")
print(f"   Size: {len(df_final):,} rows × {len(df_final.columns)} columns")

# ----------------------------
# STEP 4: Summary statistics
# ----------------------------
print("\n" + "="*70)
print("FEATURE ENGINEERING SUMMARY")
print("="*70)

print("\n📊 BUSINESS METRICS:")
print(f"   Total Customers: {len(df_final):,}")
print(f"   Churned Customers: {df_final['Churn'].sum():,} ({df_final['Churn'].mean()*100:.1f}%)")
print(f"   Total MRR: ${df_final['MRR'].sum():,.2f}")
print(f"   Total Revenue Lost to Churn: ${df_final['CLV_Lost'].sum():,.2f}")

print("\n🎯 CUSTOMER SEGMENTATION:")
print(f"   Health - Critical/Churning: {len(df_final[df_final['Health_Category'].isin(['Critical', 'Churning'])]):,}")
print(f"   Engagement - Low: {len(df_final[df_final['Engagement_Tier'] == 'Low']):,}")
print(f"   Payment - High Risk: {len(df_final[df_final['Payment_Risk'] == 'High Risk']):,}")
print(f"   Interaction - Stale: {len(df_final[df_final['Interaction_Recency'] == 'Stale']):,}")

print("\n" + "="*70)
print("✅ FEATURE ENGINEERING COMPLETE!")
print("="*70)
print("\nNext Steps:")
print("1. Load this data into PostgreSQL")
print("2. Write SQL queries for business analysis")
print("3. Build Power BI dashboard")

# Display first few rows
print("\nFirst 5 rows of final dataset:")
print(df_final.head())


FINAL DATA VALIDATION & EXPORT
1. Checking for missing values in engineered features...
✅ No missing values

2. Checking for infinite/invalid values...
✅ No infinite values

3. Final dataset shape:
Rows: 64,374
Columns: 28

CREATING ANALYSIS-READY DATASET

✅ Saved engineered dataset to: ../../data/processed/churn_data_engineered.csv
   Size: 64,374 rows × 24 columns

FEATURE ENGINEERING SUMMARY

📊 BUSINESS METRICS:
   Total Customers: 64,374
   Churned Customers: 30,493 (47.4%)
   Total MRR: $2,484,981.86
   Total Revenue Lost to Churn: $-13,978,896.86

🎯 CUSTOMER SEGMENTATION:
   Health - Critical/Churning: 36,911
   Engagement - Low: 16,638
   Payment - High Risk: 59,710
   Interaction - Stale: 21,361

✅ FEATURE ENGINEERING COMPLETE!

Next Steps:
1. Load this data into PostgreSQL
2. Write SQL queries for business analysis
3. Build Power BI dashboard

First 5 rows of final dataset:
   CustomerID  Age  Gender  Tenure  Usage Frequency  Support Calls  \
0           1   22  Female      2